# 19 — Grad-CAM 개선을 위한 얼굴 크롭 재학습
> **목적:** Grad-CAM이 배경/가장자리를 보는 문제 → 얼굴만 크롭된 이미지로 재학습  
> **베이스 모델:** `stage2_best.h5` (원본에서 시작, v3 아님)  
> **목표 모델:** `stage2_webcam_v4.h5`  
> **threshold:** 재학습 후 ROC curve로 재탐색 (기존 0.75 참고)

---
## 📋 체크리스트
- [ ] Cell 1: 환경 설정 & Drive 마운트
- [ ] Cell 2: cropped/ 샘플 시각화 → 얼굴 크롭 여부 육안 확인
- [ ] Cell 3: (조건부) MTCNN 재크롭 → `data/cropped_face/` 저장
- [ ] Cell 4: webcam 데이터 확인 & 경로 설정
- [ ] Cell 5: 통합 데이터셋 구성 (CelebA + webcam, 오버샘플링 1:1.4)
- [ ] Cell 6: 베이스 모델 로드 (stage2_best.h5) & Phase A 재학습 (Head만, LR=1e-3, 10 epochs)
- [ ] Cell 7: 학습 곡선 시각화
- [ ] Cell 8: webcam 세트 검증 (Live≥95%, Print≥90%, Replay≥90%)
- [ ] Cell 9: Threshold 탐색 (ROC curve)
- [ ] Cell 10: Grad-CAM 확인 (얼굴 중심부 활성화 검증)
- [ ] Cell 11: 모델 저장

---
## 📊 학습 데이터 구성
| 데이터 | 수량 | label | spoof_type |
|--------|------|-------|------------|
| webcam_live | 251장 | 0 | 0 |
| CelebA_live | 251장 | 0 | 0 |
| webcam_print | 54장 | 1 | 1 |
| webcam_replay | 50장 | 1 | 2 |
| CelebA_print | 300장 | 1 | 1 |
| CelebA_replay | 300장 | 1 | 2 |
| CelebA_mask | 300장 | 1 | 3 |

**오버샘플링:** Live : Spoof = 1 : 1.4  
**Phase A:** Head만 재학습 (LR=1e-3, epochs=10)

## Cell 1 — 환경 설정 & Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import cv2
import pickle
import json
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2

# 재현성
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# ── 경로 설정 ────────────────────────────────────────────────
BASE            = '/content/drive/MyDrive/face-anti-spoofing'
CROP_DIR        = f'{BASE}/data/cropped'        # 기존 CelebA 크롭
CROP_FACE_DIR   = f'{BASE}/data/cropped_face'   # MTCNN 재크롭 결과 (필요 시)
WEBCAM_DIR      = f'{BASE}/data/webcam'         # webcam 수집 데이터
MODEL_DIR       = f'{BASE}/models'
REPORT_DIR      = f'{BASE}/reports/phase5'

os.makedirs(REPORT_DIR, exist_ok=True)
for cat in ['live', 'print', 'replay', 'mask']:
    os.makedirs(f'{CROP_FACE_DIR}/{cat}', exist_ok=True)

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print()

# ── 현재 데이터 현황 확인 ─────────────────────────────────────
print('=== CelebA cropped/ 현황 ===')
for cat in ['live', 'print', 'replay', 'mask']:
    p = Path(f'{CROP_DIR}/{cat}')
    n = len(list(p.glob('*.jpg')) + list(p.glob('*.png'))) if p.exists() else 0
    print(f'  {cat:<8}: {n}장')

print()
print('=== webcam/ 현황 ===')
for cat in ['webcam_live', 'webcam_print', 'webcam_replay']:
    p = Path(f'{WEBCAM_DIR}/{cat}')
    n = len(list(p.glob('*.jpg')) + list(p.glob('*.png'))) if p.exists() else 0
    print(f'  {cat:<16}: {n}장')

## Cell 2 — cropped/ 샘플 시각화 (얼굴 크롭 여부 확인)

**판단 기준:**
- 얼굴이 이미지의 **70% 이상** 차지 → 재크롭 불필요 → Cell 3 건너뛰기
- 배경이 많거나 전신/반신 → 재크롭 필요 → Cell 3 실행

In [ ]:
def load_img_rgb(fp):
    """Drive 경로 안전 로드 (imdecode) → RGB 반환"""
    raw = np.fromfile(str(fp), dtype=np.uint8)
    img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

categories = ['live', 'print', 'replay', 'mask']
n_samples  = 4

fig, axes = plt.subplots(len(categories), n_samples, figsize=(16, 16))

for row, cat in enumerate(categories):
    folder = Path(f'{CROP_DIR}/{cat}')
    paths  = sorted(folder.glob('*.jpg'))[:n_samples]
    for col in range(n_samples):
        ax = axes[row][col]
        if col < len(paths):
            img = load_img_rgb(paths[col])
            if img is not None:
                ax.imshow(img)
                ax.set_title(f'{cat}\n{paths[col].name[:18]}', fontsize=7)
        ax.axis('off')

plt.suptitle('cropped/ 샘플 — 얼굴이 꽉 차 있나요?\n'
             '✅ 꽉 차 있음 → Cell 3 건너뛰기  /  ❌ 배경 많음 → Cell 3 실행',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/19_crop_sample_check.png', dpi=120, bbox_inches='tight')
plt.show()
print('저장:', f'{REPORT_DIR}/19_crop_sample_check.png')

## Cell 3 — (조건부) MTCNN 얼굴 재크롭

**Cell 2에서 배경이 많다고 판단될 때만 실행**  
이미 얼굴이 꽉 차 있으면 `NEED_RECROP = False`로 두고 실행 → CROP_FACE_DIR이 CROP_DIR로 대체됨

In [ ]:
# ⚠️ Cell 2 확인 후 설정
NEED_RECROP = True   # 배경이 많으면 True / 이미 얼굴 크롭이면 False

if not NEED_RECROP:
    CROP_FACE_DIR = CROP_DIR
    print(f'재크롭 불필요 → CROP_FACE_DIR = {CROP_FACE_DIR}')

else:
    print('MTCNN 설치 중...')
    os.system('pip install mtcnn -q')
    from mtcnn import MTCNN
    detector = MTCNN()
    print('MTCNN 로드 완료')

    def crop_face_mtcnn(img_bgr, margin=0.2, target_size=224):
        """MTCNN 얼굴 검출 후 마진 포함 크롭 → (224,224) BGR"""
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(img_rgb)
        if not results:
            return None
        # 가장 큰 얼굴 선택
        best = max(results, key=lambda r: r['box'][2] * r['box'][3])
        x, y, w, h = best['box']
        H, W = img_bgr.shape[:2]
        mx, my = int(w * margin), int(h * margin)
        x1 = max(0, x - mx)
        y1 = max(0, y - my)
        x2 = min(W, x + w + mx)
        y2 = min(H, y + h + my)
        face = img_bgr[y1:y2, x1:x2]
        return cv2.resize(face, (target_size, target_size))

    def center_crop_fallback(img_bgr, target_size=224):
        """MTCNN 실패 시 center square crop"""
        H, W = img_bgr.shape[:2]
        s = min(H, W)
        cy, cx = H // 2, W // 2
        crop = img_bgr[cy - s//2 : cy + s//2, cx - s//2 : cx + s//2]
        return cv2.resize(crop, (target_size, target_size))

    total_ok = 0
    total_fallback = 0

    for cat in ['live', 'print', 'replay', 'mask']:
        src = Path(f'{CROP_DIR}/{cat}')
        dst = Path(f'{CROP_FACE_DIR}/{cat}')
        dst.mkdir(parents=True, exist_ok=True)

        paths = sorted(src.glob('*.jpg')) + sorted(src.glob('*.png'))
        ok, fallback = 0, 0

        for p in paths:
            out = dst / p.name
            if out.exists():    # 이미 처리된 파일 스킵
                ok += 1
                continue
            img = cv2.imread(str(p))
            if img is None:
                continue
            face = crop_face_mtcnn(img)
            if face is not None:
                cv2.imwrite(str(out), face)
            else:
                cv2.imwrite(str(out), center_crop_fallback(img))
                fallback += 1
            ok += 1

        total_ok += ok
        total_fallback += fallback
        print(f'  [{cat}] {ok}장 완료 (MTCNN 실패→center 대체: {fallback}장)')

    print(f'\n✅ 재크롭 완료: {total_ok}장 (fallback: {total_fallback}장)')

## Cell 4 — webcam 데이터 확인 & 경로 설정

In [ ]:
# ── webcam 경로 (실제 폴더 구조에 맞게 수정) ─────────────────
WEBCAM_LIVE_DIR   = f'{WEBCAM_DIR}/webcam_live'
WEBCAM_PRINT_DIR  = f'{WEBCAM_DIR}/webcam_print'
WEBCAM_REPLAY_DIR = f'{WEBCAM_DIR}/webcam_replay'

def get_img_paths(folder):
    p = Path(folder)
    if not p.exists():
        print(f'  ⚠️  경로 없음: {folder}')
        return []
    return sorted(list(p.glob('*.jpg')) + list(p.glob('*.png')))

wl_paths = get_img_paths(WEBCAM_LIVE_DIR)
wp_paths = get_img_paths(WEBCAM_PRINT_DIR)
wr_paths = get_img_paths(WEBCAM_REPLAY_DIR)

print('=== webcam 데이터 현황 ===')
print(f'  webcam_live   : {len(wl_paths)}장  (목표 251)')
print(f'  webcam_print  : {len(wp_paths)}장   (목표  54)')
print(f'  webcam_replay : {len(wr_paths)}장   (목표  50)')

# 샘플 시각화 (각 1장)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, paths, title in zip(
    axes,
    [wl_paths, wp_paths, wr_paths],
    ['webcam_live', 'webcam_print', 'webcam_replay']
):
    if paths:
        img = load_img_rgb(paths[0])
        if img is not None:
            ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/19_webcam_sample.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 5 — 통합 데이터셋 구성 (CelebA + webcam)

- Live : Spoof = 1 : 1.4 오버샘플링
- Train 80% / Val 20% (stratify)

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMG_SIZE      = 224

def read_and_preprocess(fp):
    """경로 → (224,224,3) float32 정규화 이미지. 실패 시 None"""
    raw = np.fromfile(str(fp), dtype=np.uint8)
    img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return (img - IMAGENET_MEAN) / IMAGENET_STD

def load_set(paths, label_binary, label_spoof, max_n=None, shuffle=True):
    """(이미지 배열, bin_labels, spoof_labels) 반환"""
    if shuffle:
        paths = list(paths)
        random.shuffle(paths)
    if max_n:
        paths = paths[:max_n]
    imgs, bins, sps = [], [], []
    for fp in paths:
        img = read_and_preprocess(fp)
        if img is not None:
            imgs.append(img)
            bins.append(label_binary)
            sps.append(label_spoof)
    return imgs, bins, sps

# ── 각 세트 로드 ──────────────────────────────────────────────
print('데이터 로드 중...')

# Live (label=0, spoof=0)
cl_imgs, cl_bins, cl_sps = load_set(Path(f'{CROP_FACE_DIR}/live').glob('*.jpg'), 0, 0, max_n=251)
wl_imgs, wl_bins, wl_sps = load_set(wl_paths, 0, 0, max_n=251)
print(f'  CelebA_live  : {len(cl_imgs)}장')
print(f'  webcam_live  : {len(wl_imgs)}장')

# Spoof (label=1)
cp_imgs, cp_bins, cp_sps = load_set(Path(f'{CROP_FACE_DIR}/print').glob('*.jpg'),  1, 1, max_n=300)
cr_imgs, cr_bins, cr_sps = load_set(Path(f'{CROP_FACE_DIR}/replay').glob('*.jpg'), 1, 2, max_n=300)
cm_imgs, cm_bins, cm_sps = load_set(Path(f'{CROP_FACE_DIR}/mask').glob('*.jpg'),   1, 3, max_n=300)
wp_imgs, wp_bins, wp_sps = load_set(wp_paths, 1, 1)
wr_imgs, wr_bins, wr_sps = load_set(wr_paths, 1, 2)
print(f'  CelebA_print : {len(cp_imgs)}장')
print(f'  CelebA_replay: {len(cr_imgs)}장')
print(f'  CelebA_mask  : {len(cm_imgs)}장')
print(f'  webcam_print : {len(wp_imgs)}장')
print(f'  webcam_replay: {len(wr_imgs)}장')

# ── 합치기 ───────────────────────────────────────────────────
all_live_imgs  = cl_imgs + wl_imgs
all_live_bins  = cl_bins + wl_bins
all_live_sps   = cl_sps  + wl_sps

all_spoof_imgs = cp_imgs + cr_imgs + cm_imgs + wp_imgs + wr_imgs
all_spoof_bins = cp_bins + cr_bins + cm_bins + wp_bins + wr_bins
all_spoof_sps  = cp_sps  + cr_sps  + cm_sps  + wp_sps  + wr_sps

# ── 오버샘플링 1:1.4 ──────────────────────────────────────────
target_spoof = int(len(all_live_imgs) * 1.4)
if len(all_spoof_imgs) < target_spoof:
    idx = list(range(len(all_spoof_imgs)))
    idx_os = random.choices(idx, k=target_spoof)
    all_spoof_imgs = [all_spoof_imgs[i] for i in idx_os]
    all_spoof_bins = [all_spoof_bins[i] for i in idx_os]
    all_spoof_sps  = [all_spoof_sps[i]  for i in idx_os]
else:
    idx = list(range(len(all_spoof_imgs)))
    random.shuffle(idx)
    idx = idx[:target_spoof]
    all_spoof_imgs = [all_spoof_imgs[i] for i in idx]
    all_spoof_bins = [all_spoof_bins[i] for i in idx]
    all_spoof_sps  = [all_spoof_sps[i]  for i in idx]

# ── 최종 배열 ────────────────────────────────────────────────
X_all = np.array(all_live_imgs + all_spoof_imgs, dtype=np.float32)
y_bin = np.array(all_live_bins + all_spoof_bins, dtype=np.float32)
y_sp  = np.array(all_live_sps  + all_spoof_sps,  dtype=np.int32)

# shuffle
shuffle_idx = np.random.permutation(len(X_all))
X_all, y_bin, y_sp = X_all[shuffle_idx], y_bin[shuffle_idx], y_sp[shuffle_idx]

print(f'\n=== 최종 데이터셋 ===')
print(f'  총 샘플  : {len(X_all)}')
print(f'  Live     : {(y_bin==0).sum()}')
print(f'  Spoof    : {(y_bin==1).sum()}  (비율 1:{(y_bin==1).sum()/(y_bin==0).sum():.2f})')
print(f'  Shape    : {X_all.shape}')

# ── Train / Val 분할 (8:2) ────────────────────────────────────
idx_tr, idx_va = train_test_split(
    np.arange(len(X_all)), test_size=0.2, random_state=42, stratify=y_bin
)

X_tr, X_va   = X_all[idx_tr], X_all[idx_va]
y_bin_tr, y_bin_va = y_bin[idx_tr], y_bin[idx_va]
y_sp_tr,  y_sp_va  = y_sp[idx_tr],  y_sp[idx_va]

print(f'\n  Train: {len(X_tr)} / Val: {len(X_va)}')
print(f'  Train Live/Spoof: {(y_bin_tr==0).sum()} / {(y_bin_tr==1).sum()}')

## Cell 6 — 베이스 모델 로드 & Phase A 재학습 (Head만)

- 베이스: `stage2_best.h5`
- **backbone 완전 동결**
- Head 레이어만 LR=1e-3으로 10 epochs 학습

In [ ]:
BASE_MODEL_PATH = f'{MODEL_DIR}/stage2_best.h5'
SAVE_PATH       = f'{MODEL_DIR}/stage2_webcam_v4.h5'

# ── 베이스 모델 로드 ──────────────────────────────────────────
print('베이스 모델 로드:', BASE_MODEL_PATH)
base_model = tf.keras.models.load_model(BASE_MODEL_PATH)
base_model.summary(line_length=90)

# ── Backbone 완전 동결 ────────────────────────────────────────
# MobileNetV2 레이어 이름 확인 후 동결
frozen_count = 0
for layer in base_model.layers:
    if 'mobilenetv2' in layer.name.lower() or 'mobilenet' in layer.name.lower():
        layer.trainable = False
        frozen_count += 1

# backbone 레이어 이름 못 찾으면 전체 동결 후 마지막 몇 개만 학습
if frozen_count == 0:
    print('⚠️  MobileNetV2 레이어명 자동 감지 실패 → 수동 동결 진행')
    # 마지막 4개 레이어(헤드)만 학습
    for layer in base_model.layers[:-4]:
        layer.trainable = False
    for layer in base_model.layers[-4:]:
        layer.trainable = True

trainable_layers = [l.name for l in base_model.layers if l.trainable]
print(f'\n학습 가능 레이어 ({len(trainable_layers)}개):')
for name in trainable_layers:
    print(f'  - {name}')

# ── 컴파일 ────────────────────────────────────────────────────
base_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss={
        'binary': 'binary_crossentropy',
        'spoof' : 'sparse_categorical_crossentropy'
    },
    loss_weights={'binary': 0.7, 'spoof': 0.3},
    metrics={
        'binary': ['accuracy'],
        'spoof' : ['accuracy']
    }
)

# ── Callbacks ────────────────────────────────────────────────
cb_list = [
    callbacks.ModelCheckpoint(
        SAVE_PATH,
        monitor='val_binary_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_binary_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# ── Phase A: Head 재학습 ──────────────────────────────────────
print('\n=== Phase A: Head 재학습 (backbone frozen) ===')
history = base_model.fit(
    x=X_tr,
    y={'binary': y_bin_tr, 'spoof': y_sp_tr},
    validation_data=(X_va, {'binary': y_bin_va, 'spoof': y_sp_va}),
    epochs=10,
    batch_size=32,
    callbacks=cb_list,
    verbose=1
)

print('\n✅ Phase A 완료')
print(f'   저장: {SAVE_PATH}')

## Cell 7 — 학습 곡선 시각화

In [ ]:
hist = history.history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Binary Accuracy
axes[0].plot(hist['binary_accuracy'],     label='Train Binary Acc')
axes[0].plot(hist['val_binary_accuracy'], label='Val Binary Acc')
axes[0].set_title('Binary Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].set_ylim(0.5, 1.01)
axes[0].grid(alpha=0.4)

# Spoof Type Accuracy
axes[1].plot(hist['spoof_accuracy'],     label='Train Spoof Acc')
axes[1].plot(hist['val_spoof_accuracy'], label='Val Spoof Acc')
axes[1].set_title('Spoof Type Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].set_ylim(0.5, 1.01)
axes[1].grid(alpha=0.4)

# Total Loss
axes[2].plot(hist['loss'],     label='Train Loss')
axes[2].plot(hist['val_loss'], label='Val Loss')
axes[2].set_title('Total Loss')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.4)

plt.suptitle('Phase A 학습 곡선 — stage2_webcam_v4', fontsize=13)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/19_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장:', f'{REPORT_DIR}/19_training_curves.png')

## Cell 8 — webcam 세트 검증

**합격 기준**
| 세트 | 목표 |
|------|------|
| webcam_live | REAL ≥ 95% |
| webcam_print | FAKE ≥ 90% |
| webcam_replay | FAKE ≥ 90% |

In [ ]:
# 최적 모델 로드
best_model = tf.keras.models.load_model(SAVE_PATH)
THRESHOLD  = 0.75   # 기존 참고값, Cell 9에서 재탐색

def evaluate_webcam_set(paths, label_binary, set_name, threshold=THRESHOLD):
    """webcam 세트 전체 추론 후 정확도 반환"""
    if not paths:
        print(f'  ⚠️  {set_name}: 이미지 없음')
        return None

    imgs = []
    for fp in paths:
        img = read_and_preprocess(fp)
        if img is not None:
            imgs.append(img)

    if not imgs:
        print(f'  ⚠️  {set_name}: 로드 실패')
        return None

    X = np.array(imgs, dtype=np.float32)
    preds = best_model.predict(X, batch_size=32, verbose=0)

    # 멀티 출력 모델: preds[0] = binary prob
    if isinstance(preds, (list, tuple)):
        bin_prob = preds[0].squeeze()
    else:
        bin_prob = preds.squeeze()

    pred_labels = (bin_prob >= threshold).astype(int)   # 1=Spoof
    true_labels = np.full(len(pred_labels), label_binary)
    acc = (pred_labels == true_labels).mean() * 100

    goal = 95.0 if label_binary == 0 else 90.0
    status = '✅' if acc >= goal else '❌'
    label_name = 'REAL' if label_binary == 0 else 'FAKE'
    print(f'  {status} {set_name:<20}: {label_name} {acc:.1f}%  (목표 {goal}%)')

    return {'set': set_name, 'acc': acc, 'goal': goal, 'pass': acc >= goal}

print('=== webcam 검증 (threshold={:.2f}) ==='.format(THRESHOLD))
r_live   = evaluate_webcam_set(wl_paths, 0, 'webcam_live')
r_print  = evaluate_webcam_set(wp_paths, 1, 'webcam_print')
r_replay = evaluate_webcam_set(wr_paths, 1, 'webcam_replay')

results_webcam = [r for r in [r_live, r_print, r_replay] if r is not None]
all_pass = all(r['pass'] for r in results_webcam)
print()
print('전체 합격:', '✅ PASS' if all_pass else '❌ FAIL — threshold 재조정 필요')

## Cell 9 — Threshold 탐색 (ROC curve)

In [ ]:
# Val 세트 전체로 ROC
val_preds = best_model.predict(X_va, batch_size=32, verbose=0)
if isinstance(val_preds, (list, tuple)):
    val_bin_prob = val_preds[0].squeeze()
else:
    val_bin_prob = val_preds.squeeze()

fpr, tpr, thresholds = roc_curve(y_bin_va, val_bin_prob)
auc_score = roc_auc_score(y_bin_va, val_bin_prob)

# Youden's J = TPR - FPR 최대화 지점
j_scores = tpr - fpr
best_idx  = np.argmax(j_scores)
best_thr  = thresholds[best_idx]

print(f'ROC-AUC    : {auc_score:.4f}')
print(f'최적 Threshold (Youden J): {best_thr:.4f}')
print(f'  → TPR: {tpr[best_idx]:.4f} / FPR: {fpr[best_idx]:.4f}')

# ROC 시각화
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2980b9', lw=2, label=f'ROC (AUC={auc_score:.4f})')
plt.scatter(fpr[best_idx], tpr[best_idx], color='red', zorder=5,
            label=f'최적 Threshold={best_thr:.3f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('FPR (FAR)', fontsize=12)
plt.ylabel('TPR', fontsize=12)
plt.title('ROC Curve — stage2_webcam_v4', fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/19_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# 최적 threshold로 webcam 재검증
print(f'\n=== 최적 Threshold({best_thr:.3f})로 재검증 ===')
evaluate_webcam_set(wl_paths, 0, 'webcam_live',    threshold=best_thr)
evaluate_webcam_set(wp_paths, 1, 'webcam_print',   threshold=best_thr)
evaluate_webcam_set(wr_paths, 1, 'webcam_replay',  threshold=best_thr)

FINAL_THRESHOLD = best_thr
print(f'\n→ 최종 사용 threshold: {FINAL_THRESHOLD:.4f}')

## Cell 10 — Grad-CAM 확인 (얼굴 중심부 활성화 검증)

**확인 포인트:** 히트맵이 배경/가장자리가 아닌 **얼굴 중심부(코·눈·피부)**를 활성화하는지

In [ ]:
# ── Grad-CAM 구현 ─────────────────────────────────────────────
def get_gradcam_heatmap(model, img_array, target_layer_name=None, output_head='binary'):
    """
    img_array : (1, 224, 224, 3) 정규화 텐서
    반환: (224, 224) 히트맵 [0,1]
    """
    # target_layer 자동 탐색
    if target_layer_name is None:
        for layer in reversed(model.layers):
            if len(layer.output_shape) == 4:   # Conv layer
                target_layer_name = layer.name
                break

    # Grad-CAM 서브모델
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output,
                 model.output if not isinstance(model.output, (list, tuple))
                 else model.output[0]]
    )

    with tf.GradientTape() as tape:
        inputs     = tf.cast(img_array, tf.float32)
        conv_out, preds = grad_model(inputs)
        # sigmoid 직전 logit 기반 (포화 문제 방지)
        loss = preds[:, 0]

    grads    = tape.gradient(loss, conv_out)
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam      = conv_out[0] @ pooled[..., tf.newaxis]
    cam      = tf.squeeze(cam)
    cam      = tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-8)
    heatmap  = cv2.resize(cam.numpy(), (224, 224))
    return heatmap


def overlay_heatmap(img_norm, heatmap, alpha=0.5):
    """정규화 이미지 + 히트맵 오버레이 → RGB uint8"""
    # 역정규화
    img = (img_norm * IMAGENET_STD + IMAGENET_MEAN)
    img = np.clip(img * 255, 0, 255).astype(np.uint8)
    # heatmap 컬러맵 적용
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    superimposed = np.uint8(img * (1 - alpha) + jet * alpha)
    return superimposed


# ── 시각화: webcam_live 3장 + webcam_replay 3장 ──────────────
sample_sets = [
    (wl_paths[:3], 'webcam_live (REAL)',   0),
    (wp_paths[:3], 'webcam_print (FAKE)',  1),
    (wr_paths[:3], 'webcam_replay (FAKE)', 1),
]

fig, axes = plt.subplots(3, 6, figsize=(22, 11))

# target_layer 이름 찾기
target_layer = None
for layer in reversed(best_model.layers):
    if len(layer.output_shape) == 4:
        target_layer = layer.name
        break
print('Grad-CAM target layer:', target_layer)

for row, (paths, title, _) in enumerate(sample_sets):
    for col_offset, fp in enumerate(paths[:3]):
        img_norm = read_and_preprocess(fp)
        if img_norm is None:
            continue
        img_batch = np.expand_dims(img_norm, 0)

        # 원본
        orig_col = col_offset * 2
        orig_rgb = np.uint8(np.clip(
            (img_norm * IMAGENET_STD + IMAGENET_MEAN) * 255, 0, 255
        ))
        axes[row][orig_col].imshow(orig_rgb)
        axes[row][orig_col].set_title(f'{title}\n원본', fontsize=7)
        axes[row][orig_col].axis('off')

        # Grad-CAM
        try:
            heatmap  = get_gradcam_heatmap(best_model, img_batch, target_layer)
            overlay  = overlay_heatmap(img_norm, heatmap)
            axes[row][orig_col + 1].imshow(overlay)
            axes[row][orig_col + 1].set_title('Grad-CAM', fontsize=7)
        except Exception as e:
            axes[row][orig_col + 1].set_title(f'오류: {e}', fontsize=6)
        axes[row][orig_col + 1].axis('off')

plt.suptitle('Grad-CAM 검증 — 얼굴 중심부 활성화 확인\n'
             '빨간 영역이 코/눈/피부 → ✅  배경/가장자리 → ❌',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/19_gradcam_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장:', f'{REPORT_DIR}/19_gradcam_check.png')

## Cell 11 — 모델 저장 & 요약

In [ ]:
# 최종 결과 요약 저장
summary = {
    'model'     : 'stage2_webcam_v4.h5',
    'base_model': 'stage2_best.h5',
    'phase'     : 'A (head only, backbone frozen)',
    'threshold' : float(FINAL_THRESHOLD),
    'roc_auc'   : float(auc_score),
    'webcam_results': {
        r['set']: {'acc': round(r['acc'], 2), 'pass': r['pass']}
        for r in results_webcam
    }
}

summary_path = f'{REPORT_DIR}/19_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('=' * 60)
print('  19 — 재학습 결과 요약')
print('=' * 60)
print(f'  모델 저장     : {SAVE_PATH}')
print(f'  threshold     : {FINAL_THRESHOLD:.4f}')
print(f'  ROC-AUC       : {auc_score:.4f}')
print()
print('  webcam 검증:')
for r in results_webcam:
    status = '✅' if r['pass'] else '❌'
    print(f'    {status} {r["set"]:<22}: {r["acc"]:.1f}%  (목표 {r["goal"]}%)')
print()
all_pass = all(r['pass'] for r in results_webcam)
print('  최종 판정:', '✅ PASS → 발표 사용 가능' if all_pass else '❌ FAIL → threshold 조정 또는 Phase B 진행')
print()
print('  저장 파일:')
print(f'    {SAVE_PATH}')
print(f'    {summary_path}')
print(f'    {REPORT_DIR}/19_*.png')
print('=' * 60)
print()
print('>>> 다음 단계:')
print('  ✅ PASS → xai_explainer.py FINAL_THRESHOLD 업데이트 후 발표 준비')
print('  ❌ FAIL → Phase B: backbone 상위 30 레이어 unfreeze (LR=1e-4, 10 epochs)')